# NeMo Gym + Fireworks `async_rl_loop`: Standalone End-to-End Walkthrough

Run this notebook top to bottom -- **Run All** -- with nothing pre-cloned or pre-installed,
to reproduce a **confirmed-working, live-verified** GRPO reinforcement-fine-tuning run where
**NeMo Gym controls the RL environment** (task generation, agent harness, verification) and
**Fireworks AI** runs training and inference, wired together through Fireworks' own
`training.recipes.async_rl_loop.main()` framework.

**Ownership split:**
- **NeMo Gym owns**: all 3 servers (resources/verifier, agent harness, model server), the
  dataset (`example_multi_step` -- a genuine multi-turn tool-calling task), task
  orchestration, and verification.
- **Fireworks owns**: Dedicated trainer + inference deployment lifecycle, rollout
  fan-out/admission, GRPO group assembly, forward/backward, the optimizer, sampler hotload,
  and checkpointing -- all via `async_rl_loop.main()`.

**Model**: `accounts/fireworks/models/qwen3p5-27b` (Qwen 3.5 27B dense), LoRA rank 8,
training shape `accounts/fireworks/trainingShapes/qwen3p5-27b-64k-lora`.

**Live-verified result** (2026-08-26): 3 real GRPO optimizer steps, 20 rollouts (9 successes
/ 11 failures -- genuine mixed reward signal), clean automatic teardown. Design decisions
and known issues are documented inline in the relevant section below; if something here
behaves differently against a newer NeMo Gym / Fireworks SDK version, that's the place to
start looking.

**What you actually need before starting:**
- A **Fireworks API key** (`FIREWORKS_API_KEY`) from a billing-enabled account -- prompted
  for via `getpass` below, never hardcoded.
- **`git`** on `PATH` (to clone the two repos). This is the one thing this notebook cannot
  install for you.
- `uv` is auto-installed if missing (§1).
- **One manual, one-time step (§3)**: apply the two patch files shipped next to this notebook
  (`nemo_gym_multistep_new_files.patch`, `inference_provider_rollout_id.patch`) via `git apply`.
  These carry this session's actual working code -- it was never pushed upstream to either
  repo, so a fresh clone doesn't have it. §3 explains exactly what each patch does and why.
  Everything else -- both venvs, the NeMo Gym environment, the training run itself -- is fully
  automated in the cells below.

**This provisions a real Fireworks Dedicated trainer (3x B200) + inference deployment (~1x
B200) once you flip the safety switch below -- roughly $40-52/hr combined while both are up.**
`cleanup_on_exit=True` plus the async loop's own circuit breaker reliably tear both down
within minutes of any failure or on natural completion; the confirmed-working run cost about
$10-12 for ~13 minutes of wall-clock. On macOS the training cell runs under `caffeinate -i` so
a sleeping machine can't turn a short run into a long, expensive one -- this happened once
during development.

## 0. Safety switch

Leave this `False` on your first read-through. Every cell that spends money (the training
run) checks this flag and prints a skip message instead of running if it's `False`. Flip to
`True` only when you're ready.

In [ ]:
RUN_LIVE_CALLS = False  # flip to True when you're ready to spend money

def guarded(label):
    if not RUN_LIVE_CALLS:
        print(f"[skipped -- RUN_LIVE_CALLS is False] would run: {label}")
    return RUN_LIVE_CALLS

## 1. Prerequisites check (and auto-install `uv` if missing)

Read-only except for the `uv` install (a single official-installer script run, no other
side effects). `git` must already be on `PATH` -- this notebook can't install that for you.

In [ ]:
import shutil
import subprocess
import sys
import platform
import os
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

assert shutil.which("git"), "git is required and not on PATH -- install it before continuing."
print("git:", shutil.which("git"))

# NeMo Gym's own CLI builds shell commands like `cd {dir_path} && ...` without
# quoting the path (nemo_gym/cli/setup_command.py, nemo_gym/cli/env.py) -- a
# working directory with spaces breaks every server it launches with a
# confusing "cd: too many arguments" / "Process ... finished unexpectedly"
# failure. Not something this notebook can patch around; fail fast here
# instead of downstream in a Ray traceback.
assert " " not in str(Path.cwd()), (
    f"Run this notebook from a path with no spaces -- got {Path.cwd()!r}. "
    "NeMo Gym's own CLI does not quote paths in the shell commands it spawns "
    "servers with, and a space in the path breaks every server it launches."
)

if not shutil.which("uv"):
    print("uv not found -- installing via the official installer...")
    subprocess.run(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, check=True,
    )
    # The installer places uv in ~/.local/bin, which may not be on this process's PATH yet.
    local_bin = str(Path.home() / ".local" / "bin")
    if local_bin not in os.environ.get("PATH", "") and Path(local_bin).exists():
        os.environ["PATH"] = local_bin + os.pathsep + os.environ.get("PATH", "")

uv_path = shutil.which("uv")
assert uv_path, "uv still not found after install -- add ~/.local/bin to PATH and re-run this cell."
print("uv:", uv_path)
out = subprocess.run(["uv", "--version"], capture_output=True, text=True)
print(" ", out.stdout.strip())


## 2. Clone NeMo Gym and the Fireworks training cookbook

Both idempotent -- skip cloning if the directory already exists next to this notebook.

In [ ]:
HERE = Path.cwd()
NEMO_GYM_DIR = HERE / "nemo-gym"
COOKBOOK_DIR = HERE / "cookbook"

if NEMO_GYM_DIR.exists():
    print(f"Already present: {NEMO_GYM_DIR}")
else:
    subprocess.run(
        ["git", "clone", "https://github.com/NVIDIA-NeMo/Gym.git", str(NEMO_GYM_DIR)],
        check=True,
    )

if COOKBOOK_DIR.exists():
    print(f"Already present: {COOKBOOK_DIR}")
else:
    subprocess.run(
        ["git", "clone", "https://github.com/fw-ai/cookbook.git", str(COOKBOOK_DIR)],
        check=True,
    )

print(sorted(p.name for p in NEMO_GYM_DIR.iterdir())[:10])
print(sorted(p.name for p in COOKBOOK_DIR.iterdir())[:10])

## 3. Apply this session's changes -- required, one-time, manual

**You need to do this yourself before continuing.** The `nemo_gym_multistep` example
(`proxy.py`, `train.py`) and a small `inference_provider` patch were built and live-verified
in the development session behind this notebook, but were never pushed upstream to
`fw-ai/cookbook` or `NVIDIA-NeMo/Gym` -- a fresh clone of either repo (as you just did in §2)
does not include them. This notebook ships two patch files next to it,
`nemo_gym_multistep_new_files.patch` and `inference_provider_rollout_id.patch` -- both
verified to apply cleanly against a fresh clone of each repo (checked at the time of writing;
re-verify with `git apply --check` if either upstream repo has moved since).

Run, from a terminal, from wherever this notebook lives:

```bash
cd cookbook && git apply ../nemo_gym_multistep_new_files.patch && cd ..
cd nemo-gym && git apply ../inference_provider_rollout_id.patch && cd ..
```

**What each patch does and why:**

**`nemo_gym_multistep_new_files.patch`** adds three new files under
`cookbook/training/examples/rl/nemo_gym_multistep/`:
- **`proxy.py`** -- `TinkerRecordingProxy`, a local OpenAI-compatible HTTP server. NeMo Gym's
  model server is pointed at it instead of the real Fireworks API. It forwards each request to
  the live Fireworks Dedicated deployment via `DeploymentSampler`, and records exact
  prompt/completion token ids + logprobs per turn -- the data `async_rl_loop.main()` needs to
  actually train. Trajectory tracking (matching an incoming request to the right in-progress
  conversation) uses content-hash prefix matching rather than exact-dict-equality, specifically
  because NeMo Gym's agent reconstructs messages fresh from its own Responses API state every
  turn, so exact-equality matching hard-crashed on turn 2 in testing.
- **`train.py`** -- calls `training.recipes.async_rl_loop.main()` (Fireworks' own async RL
  framework) with a `rollout_fn` that POSTs directly to the live NeMo Gym agent's `/run`
  endpoint, one call per sample, then drains the matching `proxy.py` session into a training
  sample for the recipe.
- **`__init__.py`** -- empty, makes the directory an importable package.

**`inference_provider_rollout_id.patch`** modifies one existing file,
`nemo-gym/responses_api_models/inference_provider/app.py`, adding a small middleware
(`_CaptureRolloutIdMiddleware`). NeMo Gym's own per-rollout correlation mechanism
(`current_rollout_id()`) turns out to only be wired up on *resources servers*, never on model
servers -- calling it from `inference_provider` (a model server) silently always returns
`None`. Without this patch, every concurrent rollout collides into one shared proxy session
and corrupts each other's conversation state. The middleware recovers the correlation id
directly from the `/ng-rollout/<id>` URL path prefix before NeMo Gym's own middleware strips
it, and threads it into the outbound OpenAI `user` field so `proxy.py` can key sessions on it
correctly.

The verification cell below confirms both patches were applied before you continue.

In [ ]:
nemo_gym_multistep_dir = COOKBOOK_DIR / "training" / "examples" / "rl" / "nemo_gym_multistep"
app_py_path = NEMO_GYM_DIR / "responses_api_models" / "inference_provider" / "app.py"

missing = []
if not (nemo_gym_multistep_dir / "proxy.py").exists():
    missing.append("cookbook: nemo_gym_multistep_new_files.patch not applied (proxy.py missing)")
if not (nemo_gym_multistep_dir / "train.py").exists():
    missing.append("cookbook: nemo_gym_multistep_new_files.patch not applied (train.py missing)")
if "_CaptureRolloutIdMiddleware" not in app_py_path.read_text():
    missing.append("nemo-gym: inference_provider_rollout_id.patch not applied (app.py unpatched)")

if missing:
    raise SystemExit(
        "Required patches not applied yet -- see the instructions above.\n  " + "\n  ".join(missing)
    )
print("Both patches applied correctly.")

## 4. Create the two venvs

Runs `uv sync` for both repos right here -- no separate terminal needed. Each repo gets its
own `.venv` (different dependency sets: NeMo Gym's own servers vs. the Fireworks training
SDK). This can take a few minutes on first run (downloading a pinned Python + all deps).

In [ ]:
def run_streamed(cmd, cwd):
    print(f"$ {' '.join(cmd)}  (cwd={cwd})")
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"{cmd[0]} failed in {cwd} (exit {result.returncode})")

# Best-effort -- some uv installs (e.g. via a system package manager) don't support self
# update; that's fine, sync still works against whatever version is present.
subprocess.run(["uv", "self", "update"], cwd=NEMO_GYM_DIR, capture_output=True)

run_streamed(["uv", "sync"], cwd=NEMO_GYM_DIR)
run_streamed(["uv", "sync"], cwd=COOKBOOK_DIR / "training")

assert (NEMO_GYM_DIR / ".venv" / "bin" / "gym").exists(), "nemo-gym/.venv missing the `gym` CLI after uv sync"
assert (COOKBOOK_DIR / "training" / ".venv" / "bin" / "python3").exists(), "cookbook/training/.venv missing after uv sync"
print("Both venvs ready.")

## 5. Fireworks credentials

Never hardcode your API key in a notebook cell -- it'll end up in the saved `.ipynb`
output/metadata and possibly in version control. Use `getpass` so it's never echoed or
persisted in the notebook file. The account id is auto-resolved from the key itself right
after -- you don't need to already know it.

In [ ]:
import getpass
import json as _json
import urllib.request

if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("FIREWORKS_API_KEY: ")

def _fw_get(path):
    req = urllib.request.Request(
        f"https://api.fireworks.ai/v1{path}",
        headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"},
    )
    with urllib.request.urlopen(req) as resp:
        return _json.load(resp)

accounts = _fw_get("/accounts")["accounts"]
assert accounts, "No accounts visible for this API key -- check it's valid and billing-enabled."
FIREWORKS_ACCOUNT_ID = accounts[0]["name"].rsplit("/", 1)[-1]
os.environ["FIREWORKS_ACCOUNT_ID"] = FIREWORKS_ACCOUNT_ID
print("Resolved account:", FIREWORKS_ACCOUNT_ID)

## 6. Architecture

```
                        NeMo Gym owns this box                     Fireworks owns this box
        +-------------------------------------------------+   +--------------------------------+
        |  resources server   agent harness   model server |   |  Dedicated trainer (GRPO)      |
        |  (verifier, tools)  (SimpleAgent,    (inference_  |   |  Dedicated inference deployment |
        |                      multi-turn      provider,   |<--+  (hotloaded every step)         |
        |                      tool loop)      patched)    |   |                                  |
        +-------------------------------------------------+   +--------------------------------+
                        |                          |
                        |  /run (direct HTTP)       |  proxy.py: TinkerRecordingProxy
                        |  from rollout_fn           |  (OpenAI-compatible HTTP server,
                        v                          v   samples from the live deployment,
              async_rl_loop.main()  <----------------  records exact tokens/logprobs)
              (GRPO groups, optimizer,
               checkpointing, hotload)
```

`train.py`'s `rollout_fn` makes one direct HTTP `/run` POST per sample straight to the live
`example_multi_step_simple_agent` harness (setting NeMo Gym's own
`_ng_task_index`/`_ng_rollout_index` correlation fields itself), then drains the matching
`proxy.py` session into a `RolloutRun`/`RolloutSample` for the recipe.

Two design points worth knowing if you extend this to a new environment:

1. **NeMo Gym's rollout-correlation contextvar (`current_rollout_id()`) is only wired up on
   resources servers, never model servers.** `inference_provider/app.py` is patched with a
   custom `_CaptureRolloutIdMiddleware` that reads the `/ng-rollout/<id>` path prefix
   directly, before NeMo Gym's own middleware strips it.
2. **`example_multi_step_simple_agent` drives conversation state via NeMo Gym's Responses
   API, not raw chat messages** -- it reconstructs a fresh chat-messages list every turn,
   which isn't guaranteed byte-identical across turns. `proxy.py` uses content-hash prefix
   matching for trajectory tracking rather than exact-dict-equality checkpointing,
   specifically because it degrades gracefully (loses that turn's training linkage) instead
   of crashing on a hash mismatch.

## 7. Preview the dataset (no cost)

`example_multi_step` is NeMo Gym's tutorial multi-turn tool-calling environment: the model
must call `get_synonym_value`/`extract_synonym_values` tools to extract the correct values
from a large embedded synonym table.

In [ ]:
RUN_DIR = COOKBOOK_DIR / "training" / "examples" / "rl" / "nemo_gym_multistep"
dataset_path = NEMO_GYM_DIR / "resources_servers" / "example_multi_step" / "data" / "example.jsonl"
rows = [_json.loads(line) for line in dataset_path.read_text().splitlines() if line.strip()]
print(f"{len(rows)} rows in {dataset_path}")

row = rows[0]
user_message = next(m for m in row["responses_create_params"]["input"] if m["role"] == "user")
print("Example user query:", user_message["content"])

## 8. See the NeMo Gym environment actually start (free, no Fireworks cost)

This is entirely local -- `gym env start` boots all 3 NeMo Gym servers (resources/verifier,
agent harness, model server) and nothing here talks to Fireworks. Confirms the environment
itself works *before* spending anything. We stop it again immediately afterward: the real
training run in the next section starts its own instance internally (via `train.py`), and
two instances would fight over the same ports.

In [ ]:
import time

# inference_provider's config interpolates ${policy_base_url} -- gym env start
# fails immediately without it, even for this no-cost readiness check (nothing
# needs to actually be listening on it yet; the model server just constructs a
# client at startup, it doesn't eagerly connect). train.py writes the real one
# pointing at TinkerRecordingProxy before its own gym env start call -- this is
# the same env.yaml shape with a placeholder URL, just to prove the environment
# itself boots.
capture_dir = NEMO_GYM_DIR / "results" / "model_call_capture"
(NEMO_GYM_DIR / "env.yaml").write_text(
    "# env.yaml -- placeholder written by this notebook's free readiness demo\n"
    "policy_base_url: http://127.0.0.1:18234/v1\n"
    'policy_api_key: "unused"\n'
    "policy_model_name: policy\n"
    "observability_enabled: true\n"
    f"model_call_capture_dir: {capture_dir}\n"
)

demo_log = RUN_DIR / "gym_env_demo.log"
with open(demo_log, "w") as f:
    demo_proc = subprocess.Popen(
        [str(NEMO_GYM_DIR / ".venv" / "bin" / "gym"), "env", "start",
         "--resources-server", "example_multi_step", "--model-type", "inference_provider", "-v"],
        cwd=NEMO_GYM_DIR, stdout=f, stderr=subprocess.STDOUT,
    )

print(f"gym env start launched (pid={demo_proc.pid}), waiting for readiness...")
deadline = time.time() + 120
ready = False
while time.time() < deadline:
    text = demo_log.read_text(errors="replace")
    if "All 3 / 3 servers ready" in text:
        ready = True
        break
    if demo_proc.poll() is not None:
        break
    time.sleep(2)

print("All 3 servers ready:", ready)
if not ready:
    print(f"--- see {demo_log} for what happened ---")

demo_proc.terminate()
try:
    demo_proc.wait(timeout=30)
except subprocess.TimeoutExpired:
    demo_proc.kill()
print("Demo environment stopped.")


## 9. Choose a model: dense or MoE

Both presets below are **live-verified** through this exact pipeline, unmodified --
switching models is just a config swap, no code changes. Uncomment the one you want to run;
leave the other commented out.

MoE trainers provision noticeably slower than dense ones (more GPUs to bring up) -- expect
the training cell to sit quietly for **~11 minutes** before the first rollout log line
appears with the MoE preset, vs. ~6-7 minutes for the dense preset. That's normal, not a
hang.

In [ ]:
# --- Dense: Qwen 3.5 27B (confirmed working) ---
MODEL_CONFIG = {
    "base_model": "accounts/fireworks/models/qwen3p5-27b",
    "tokenizer_model": "Qwen/Qwen3.5-27B",
    "training_shape_id": "accounts/fireworks/trainingShapes/qwen3p5-27b-64k-lora",
}

# --- MoE: Qwen 3.5 35B-A3B, 35B total / 3B active params (confirmed working) ---
# MODEL_CONFIG = {
#     "base_model": "accounts/fireworks/models/qwen3p5-35b-a3b",
#     "tokenizer_model": "Qwen/Qwen3.5-35B-A3B",
#     "training_shape_id": "accounts/fireworks/trainingShapes/qwen3p5-35b-a3b-256k-lora",
# }

print("Using:", MODEL_CONFIG["base_model"])

## 10. Run the training loop (live, paid)

This is the actual command that produced the confirmed-working result. It:

1. Starts `TinkerRecordingProxy` (local HTTP server).
2. Writes `nemo-gym/env.yaml` pointing NeMo Gym's model server at the proxy.
3. Launches `gym env start` (all 3 NeMo Gym servers, local, free) and waits for readiness --
   the same step you just watched work above, now running for real.
4. Calls `async_rl_loop.main()`, which provisions the Dedicated trainer + deployment, runs
   the GRPO loop against `rollout_fn`, and tears everything down on completion or failure.

Kept small here (5 rows, `completions_per_prompt=4`) to match the confirmed-working smoke
config -- scale up `--max-rows`/`--completions-per-prompt`/`--prompt-groups-per-step` for a
real training run. Pass `--output-model-id accounts/<your-account>/models/<name>` if you want
the trained checkpoint promoted to a standalone, deployable model afterward (without it, the
checkpoint still exists -- see §10 -- but isn't promoted).

Runs under `caffeinate -i` on macOS only (guards against system sleep interrupting the run;
not applicable on Linux/Windows, where the notebook host is typically a server that doesn't
sleep).

In [ ]:
if guarded("async_rl_loop training run"):
    prefix = ["caffeinate", "-i"] if sys.platform == "darwin" else []
    cmd = prefix + [
        str(COOKBOOK_DIR / "training" / ".venv" / "bin" / "python3"),
        "-m", "training.examples.rl.nemo_gym_multistep.train",
        "--base-model", MODEL_CONFIG["base_model"],
        "--tokenizer-model", MODEL_CONFIG["tokenizer_model"],
        "--training-shape-id", MODEL_CONFIG["training_shape_id"],
        "--max-rows", "5",
        "--completions-per-prompt", "4",
        "--prompt-groups-per-step", "2",
        "--max-completion-tokens", "256",
        "--run-timeout-s", "180",
    ]
    # train.py logs via logging.basicConfig(), which defaults to stderr, not
    # stdout -- every INFO line (trainer state, rollout rewards, the trainer
    # job id, "Async RL training complete", the checkpoint name) lives there.
    # Merge streams so nothing gets silently dropped on either a successful
    # or a failed run.
    result = subprocess.run(cmd, cwd=COOKBOOK_DIR, env=os.environ, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-12000:])
    print(f"\n[exit code: {result.returncode}]")


## 11. Scaling up beyond the smoke test

Everything above runs the deliberately tiny smoke config (5 rows, 3 optimizer steps) --
enough to prove the pipeline is correct, not enough to actually improve the model. Options
below, roughly in order of effort:

**1. More steps from the same 5 rows -- zero setup, works today.**
Add `--epochs N` to the training command in §10. Each epoch cycles back through the same
rows, so this multiplies optimizer steps without needing more data -- the cheapest way to see
whether the GRPO loop actually converges on this task, even if it's ultimately data-limited.

**2. A bigger dataset for the same environment.**
`example_multi_step`'s own `README.md` references `data/train.jsonl` / `data/validation.jsonl`
as the intended larger splits, but they're gitignored and not present in a fresh clone --
only the 5-row `example.jsonl` demo file ships. You'd need to generate a larger split
yourself in the same schema (each row: `id`, `responses_create_params` with a Responses-API
`input` message list, `verifier_metadata`) and pass `--dataset-path
/path/to/your/train.jsonl`.

**3. A different, larger NeMo Gym environment entirely.**
`train.py` takes `--resources-server`/`--agent-name`/`--dataset-path` to point at any NeMo
Gym resources server + agent pair, not just `example_multi_step` (see §6, Architecture,
above). The proxy's trajectory tracking and the Fireworks-side training loop are
environment-agnostic already -- this path is untested live, but no code changes should be
required in principle.

**4. Tune the scheduling knobs** (full semantics in
`cookbook/skills/fireworks-training/references/rl-async.md`, "Five scheduling knobs"):
- `--completions-per-prompt` (must stay >= 2 -- GRPO's advantage is undefined on a
  single-sample group) -- more completions per prompt means more chances for a GRPO group to
  have surviving members with different rewards.
- `--prompt-groups-per-step` -- how many dataset rows form one optimizer batch. Larger means
  fewer, bigger, more stable optimizer steps; smaller means faster iteration.
- `--max-concurrency-rollout-sample` -- caps in-flight rollout HTTP calls against your NeMo
  Gym harness; raise this once you trust the harness handles more concurrency, to keep the
  Fireworks deployment saturated rather than idling between batches.
- `--max-head-offpolicy-versions` and pipeline-chunking aren't wired up as CLI flags yet
  (they exist on the underlying `Config` -- see `training/recipes/async_rl_loop.py`) -- a
  small `train.py` change if you need them.

**5. Keep the result.**
Pass `--output-model-id accounts/<you>/models/<name>` so the recipe promotes the final
checkpoint to a real, durable, dashboard-visible model automatically (see §12 for why this
matters -- without it, the checkpoint is only reachable via the trainer job's own id, which
is easy to lose).

**6. Budget for it.**
A longer run means many more optimizer steps, each with its own rollout/hotload round trip --
wall-clock (and cost, at ~$40-100/hr depending on shape) scales roughly linearly with steps.
Use `--wandb-entity`/`--wandb-project` to track a longer run's reward/loss curves instead of
reading raw logs, and raise `--run-timeout-s` if `--max-completion-tokens` goes up (longer
generations take proportionally longer per rollout).

## 12. Verify teardown (always safe to run, no cost)

Confirms no Dedicated deployment or trainer job is left running -- run this after any live
attempt, successful or not. `cleanup_on_exit=True` plus the async loop's circuit breaker
should always leave this clean, but this is the ground-truth check against the live API
rather than trusting the log. **If you kill the training cell manually (e.g. interrupting the
notebook kernel) instead of letting it exit on its own, this check becomes essential** --
manually stopping the underlying process can bypass graceful cleanup and leave paid resources
running; this happened once during development.

In [ ]:
deployments = _fw_get(f"/accounts/{FIREWORKS_ACCOUNT_ID}/deployments")
print("Active deployments:", deployments.get("totalSize", 0))
for dep in deployments.get("deployments", []):
    print(" -", dep.get("name"), dep.get("state"))
assert deployments.get("totalSize", 0) == 0, \
    "A deployment is still running -- delete it manually via the Fireworks console or API."
print("Clean -- nothing left running.")

## 13. Where the trained checkpoint lives

Without `--output-model-id`, the run above does **not** produce a standalone, deployable
Fireworks model -- but the trained LoRA weights aren't lost either. They exist as a
checkpoint under the (by-then-deleted) trainer job's own namespace, still queryable:

```python
from fireworks.training.sdk import FireworksClient
client = FireworksClient(api_key=os.environ["FIREWORKS_API_KEY"])
client.list_checkpoints("<trainer-job-id-from-the-log-above>")
```

The final step's checkpoint (`step-N-<session>`, `CHECKPOINT_TYPE_INFERENCE_LORA`) is the
one worth promoting if you want to actually serve or resume from it -- either re-run with
`--output-model-id` so the recipe promotes it automatically, or promote it manually via
`client.promote_checkpoint(...)`.

## 14. Reference: the confirmed-working results

For reference (no need to reproduce these numbers exactly -- RL is stochastic), this is what
the verified runs actually produced.

**Dense (`qwen3p5-27b`):**

| Step | Mean reward | Samples |
|---|---|---|
| 1 | 0.625 | 8 |
| 2 | 0.125 | 8 |
| 3 | 0.750 | 4 |

20 total rollouts: 9 scored `reward=1.0`, 11 scored `reward=0.0`.

**MoE (`qwen3p5-35b-a3b`, 35B total / 3B active):**

| Step | Mean reward | Samples |
|---|---|---|
| 1 | 0.250 | 8 |
| 2 | 0.125 | 8 |
| 3 | 0.250 | 4 |

20 total rollouts: 4 scored `reward=1.0`, 16 scored `reward=0.0` (lower hit rate than the
dense model -- plausibly a harder fit for this model/task combination, not a pipeline issue).

Both runs: the model correctly called `get_synonym_value` (x2), then `extract_synonym_values`
with the right values, then answered correctly on successful rollouts -- a genuine,
non-trivial agentic trajectory, correctly scored by NeMo Gym's real verifier. Three real
optimizer steps each, each with an actual weight hotload back into the live deployment and a
saved checkpoint on the final step. Both runs finished naturally and self-cleaned:

```
Async RL training complete: 3 steps (3 new in this run)
Scaled deployment to zero: <model>-...
Deleted trainer job: training-api-service-...
```

See `training/examples/rl/nemo_gym_multistep/{proxy.py,train.py}` in your cloned `cookbook`
checkout for the working code (note: the first MoE attempt against this shape was killed
prematurely by a too-short no-progress timeout during development -- larger training shapes
provision slower, budget accordingly if you add your own timeout around the training cell).
`train.py` also takes `--resources-server`/`--agent-name`/`--dataset-path` if you want to
point this at a different NeMo Gym environment instead of `example_multi_step`.